## 离线 阶段

In [1]:
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_core.documents import Document


def prepare_docs() -> list[Document]:
    # =============1.读取文档================
    with open('./评估.md', 'r', encoding='utf-8') as f:
        doc = f.read()

    # =============2.切分文档================
    # 切分依据，这里是按照三级标题
    # 遇到  # 开头的行，就把文字记到 metadata 的 h1 键里
    headers_to_split_on = [
        ("#", "h1"),
        ("##", "h2"),
        ("###", "h3"),
    ]

    # 创建切分器
    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on,
        strip_headers=False  # 在正文中是否去除标题，False，不去除
    )

    docs = markdown_splitter.split_text(doc)
    return docs


In [2]:
from langchain_ollama import OllamaEmbeddings
# 1.初始化向量模型，可以把文档转稠密向量
def init_ollama_emb():
    embeddings = OllamaEmbeddings(
        model="qwen3-embedding:0.6b",  # 性价比高的模型
        dimensions=1024  # 可选：减少维度以节省存储
    )
    return embeddings


In [4]:
# create_collection() # 创建collection

In [5]:
# 把文档向量化，要传入list[str]
docs = prepare_docs()
embeddings = init_ollama_emb()
vectors = embeddings.embed_documents([doc.page_content for doc in docs])

In [3]:
from pymilvus import MilvusClient
from app.core.config import settings
from pymilvus import Function, FunctionType, DataType

# 初始化客户端
client = MilvusClient(uri=settings.rag.milvus_url)
collection_name = "collection_66"
def create_collection():
    # 1.schema 声明字段
    schema = client.create_schema(
        auto_id=False,  # 不自动生成id
        enable_dynamic_field=False,  # 是否允许动态字段
    )
    # 1.1.主键字段
    schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
    # 1.2.稠密向量字段
    schema.add_field(field_name="dense", datatype=DataType.FLOAT_VECTOR, dim=1024)
    # 1.3.稀疏向量字段
    schema.add_field(field_name="sparse", datatype=DataType.SPARSE_FLOAT_VECTOR)
    # 1.4.普通元数据字段，用来做数据过滤
    schema.add_field(field_name="h2", datatype=DataType.VARCHAR, max_length=128)
    # 1.5.原始文档内容，需要被向量化
    schema.add_field(field_name="content", datatype=DataType.VARCHAR, max_length=2048,
                     enable_analyzer=True,  # 允许对该字段分词，转BM25稀疏向量
                     analyzer_params={"type": "chinese"}  # 指定分词方式，这里选择中文分词
                     )

    # 1.6.添加BM25函数，可以自动把原始文档转BM25向量存储
    bm25_function = Function(
        name="text_bm25_emb",  # Function name
        input_field_names=["content"],  # 包含原始文档数据的字段的名称
        output_field_names=["sparse"],  # 稀疏向量字段的名称
        function_type=FunctionType.BM25,  # 函数类型，这里是BM25函数
    )
    schema.add_function(bm25_function)

    # 2.添加索引
    index_params = client.prepare_index_params()
    # 2.1.稠密向量索引
    index_params.add_index(field_name="dense",  # 字段名
                           index_type="AUTOINDEX",  # 稠密向量索引方式
                           metric_type="COSINE"  # 相似度计算方式，COSINE、IP
                           )
    # 2.2.稀疏向量索引
    index_params.add_index(field_name="sparse",  # 字段名
                           index_type="SPARSE_INVERTED_INDEX",  # 稀疏向量，采用倒排索引
                           metric_type="BM25",  # 相似度计算方式，支持 BM25、IP
                           )
    # 2.3.普通字段索引
    index_params.add_index(field_name="h2", index_type="AUTOINDEX")

    # 3.创建collection
    client.create_collection(
        collection_name=collection_name,  # 只能用字母/数字/下划线，不能有 -
        schema=schema,
        index_params=index_params,
    )

In [6]:
data = [{
    "id": i + 1,
    "content": doc.page_content,
    "h2": doc.metadata.get("h2", ""),
    "dense": vectors[i],  # 稠密向量字段(长度为1024维的float数组,恰好vectors符合所以直接赋值了)
} for i, doc in enumerate(docs)]

client.insert(collection_name=collection_name, data=data)
client.flush(collection_name)  # 确保立即可见
print(client.get_collection_stats(collection_name))  # {'row_count': 21}

{'row_count': 63}


## 在线检索 阶段

In [10]:
from pymilvus import WeightedRanker
from pymilvus import AnnSearchRequest, RRFRanker
# 定义加权融合策略
ranker1 = WeightedRanker(0.5, 0.5)
# 定义RRF融合策略
ranker2 = RRFRanker(k=60)

# RRF 混合检索 和融合重排
def hybrid_search(query: str, ranker,filter_query: str = None):
    # 1.创建稠密、稀疏
    # 稠密
    query_dense_vector = embeddings.embed_query(query)
    dense_request = AnnSearchRequest(
        data=[query_dense_vector],
        anns_field="dense",
        param={"metric_type": "COSINE"},
        limit=3,
        filter=filter_query
    )
    # 稀疏
    sparse_request = AnnSearchRequest(
        data=[query],
        anns_field="sparse",
        param={"metric_type": "BM25"},
        limit=3,
        filter=filter_query
    )


    # 混合在一起
    reqs = [dense_request, sparse_request]
    # 2.融合重排
    res = client.hybrid_search(
        collection_name=collection_name,
        reqs=reqs,
        ranker=ranker,
        limit=2,
        output_fields=['content', 'h2']
    )
    #List[List[dict]]
    return reqs

In [11]:
reqs = hybrid_search("教育的负面作用是什么", ranker1)
print(reqs)

[<pymilvus.client.abstract.AnnSearchRequest object at 0x000001DDD071F6F0>, <pymilvus.client.abstract.AnnSearchRequest object at 0x000001DDD075B1D0>]
